# Causal replay table (issue #90)

Does ablating the #89 features at layer 33 change whether the Solver adopts the Critic's answer?

For each eligible intervention episode, the stored A1 and Critic feedback are replayed and the Solver's A2 is
regenerated with each SAE spliced into layer 33 at the final prompt token of the A2 prefill, under three
conditions: **no ablation** (SAE reconstruction only), **targeted ablation** (the #89 features zeroed) and
**random ablation** (frequency-matched random features, 5 draws). Acceptance = the regenerated A2 is labeled
`adopted_critic` under the Behavior v0.1.2 rules. Δ is against the same SAE's no-ablation run, paired by question.

This notebook rebuilds the table from `per_row.csv` with pandas and checks it against the run's `causal_table.csv`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from mas_sae.causal import offline

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RUN = ROOT / "results/causal_replay/20261016T215703321705Z"
LABELS = ROOT / "results/behavior/behavior_v012_official_20261011/labels.csv"  # not on GitHub (#110 is metadata-only)

cfg = json.loads((RUN / "replay_config.json").read_text())["config"]
summary = json.loads((RUN / "causal_replay_summary.json").read_text())
episodes = pd.read_csv(RUN / "episodes.csv")
rows = pd.read_csv(RUN / "per_row.csv")
print(f"{len(episodes)} episodes, {len(rows)} generations")

517 episodes, 11374 generations


## Where N = 517 comes from

N is every episode of the intervention partition with `eligible_primary = True` in the Behavior v0.1.2 labels,
not a stratified subset. The other episodes have no Solver-vs-Critic disagreement to accept or reject (the Critic
agreed with A1, refused, or gave no usable answer), so they have no acceptance outcome. #89 uses the same rule.

In [2]:
if LABELS.exists():
    labels = pd.read_csv(LABELS)
    display(pd.crosstab(labels.partition, labels.eligible_primary, margins=True))
    iv = labels[labels.partition == "intervention"]
    display(iv[~iv.eligible_primary].exclusion_reasons.value_counts().rename("excluded intervention episodes"))
    assert set(iv[iv.eligible_primary].question_id) == set(episodes.question_id)
    print(f"Eligible intervention episodes == replayed episodes: {len(episodes)}")
else:
    print(f"{LABELS} not found; the saved outputs above show the breakdown.")

eligible_primary,False,True,All
partition,,,
intervention,1474,517,1991
test,1508,484,1992
train,11759,4177,15936
validation,1777,636,2413
All,16518,5814,22332


exclusion_reasons
no_conflict                                           608
critic_refusal;a2_nonanswer                           308
a1_critic_containment                                 132
critic_unresolved                                      89
critic_refusal;a1_nonanswer;a2_nonanswer               70
critic_refusal                                         68
a2_relationship_unresolved                             57
critic_premise_rejection;a2_nonanswer                  44
a2_nonanswer                                           38
a1_nonanswer                                           19
critic_premise_rejection                               13
a1_critic_containment;a2_relationship_unresolved        8
a1_nonanswer;a2_relationship_unresolved                 6
no_conflict;a2_relationship_unresolved                  4
a1_critic_containment;a2_nonanswer                      4
no_conflict;a2_nonanswer                                2
critic_refusal;a1_nonanswer                           

Eligible intervention episodes == replayed episodes: 517


## The run is complete: every episode has every condition

In [3]:
counts = rows.groupby(["experiment", "condition"]).agg(
    episodes=("question_id", "nunique"), generations=("question_id", "size"))
display(counts)
assert (counts.episodes == len(episodes)).all()
print("Plain replay reproduces the stored A2:", summary["replay_check"])

episodes  generations
experiment     condition                               
Baseline       no_sae                  517          517
L1 SAE         no_ablation             517          517
               random_ablation         517         2585
               targeted_ablation       517          517
Pretrained SAE no_ablation             517          517
               random_ablation         517         2585
               targeted_ablation       517          517
TopK SAE       no_ablation             517          517
               random_ablation         517         2585
               targeted_ablation       517          517

Plain replay reproduces the stored A2: {'a2_exact_match_rate': 1.0, 'solver_response_match_rate': 1.0}


## The table

Per-question acceptance (random ablation is the mean over its 5 draws), then the paired bootstrap CI and
permutation p-value from `mas_sae.causal.offline.paired_effect`, the same function the pipeline uses.

In [4]:
qids = sorted(episodes.question_id)
groups = np.array(qids)

def acceptance(exp, cond):
    r = rows[(rows.experiment == exp) & (rows.condition == cond)]
    return r.groupby("question_id")["adopted"].mean().reindex(qids).to_numpy(float)

stored = rows[rows.experiment == "Baseline"].set_index("question_id")["stored_adopted"].reindex(qids)
table = [
    {"Experiment": "Baseline", "Condition": "No SAE (stored A2)", "N": len(qids), "Acceptance Rate (%)": 100 * stored.mean()},
    {"Experiment": "Baseline", "Condition": "No SAE (replayed A2)", "N": len(qids),
     "Acceptance Rate (%)": 100 * acceptance("Baseline", "no_sae").mean()},
]
for exp in ["TopK SAE", "L1 SAE", "Pretrained SAE"]:
    ref = acceptance(exp, "no_ablation")
    table.append({"Experiment": exp, "Condition": "No ablation", "N": len(qids), "Acceptance Rate (%)": 100 * ref.mean()})
    for cond, name in [("random_ablation", "Random ablation"), ("targeted_ablation", "Targeted ablation")]:
        y = acceptance(exp, cond)
        e = offline.paired_effect(y, ref, groups, cfg["n_bootstrap"], cfg["n_permutations"], cfg["seed"])
        table.append({"Experiment": exp, "Condition": name, "N": len(qids), "Acceptance Rate (%)": 100 * y.mean(),
                      "Δ vs. SAE Control (pp)": e["delta_pp"],
                      "95% CI for Δ": f"[{e['ci95_pp'][0]:.2f}, {e['ci95_pp'][1]:.2f}]", "p-value": e["p_value"]})
table = pd.DataFrame(table)
display(table.style.format({"Acceptance Rate (%)": "{:.2f}", "Δ vs. SAE Control (pp)": "{:+.2f}", "p-value": "{:.3f}"},
                           na_rep=""))

,Experiment,Condition,N,Acceptance Rate (%),Δ vs. SAE Control (pp),95% CI for Δ,p-value
0,Baseline,No SAE (stored A2),517,88.59,,,
1,Baseline,No SAE (replayed A2),517,88.59,,,
2,TopK SAE,No ablation,517,86.65,,,
3,TopK SAE,Random ablation,517,86.46,-0.19,"[-0.74, 0.23]",0.547
4,TopK SAE,Targeted ablation,517,85.88,-0.77,"[-1.55, -0.19]",0.121
5,L1 SAE,No ablation,517,85.69,,,
6,L1 SAE,Random ablation,517,85.57,-0.12,"[-0.31, 0.00]",0.495
7,L1 SAE,Targeted ablation,517,85.69,+0.00,"[-0.97, 1.16]",1.000
8,Pretrained SAE,No ablation,517,88.20,,,
9,Pretrained SAE,Random ablation,517,88.01,-0.19,"[-0.43, -0.04]",0.122


In [5]:
official = pd.read_csv(RUN / "causal_table.csv")
for col in ["Acceptance Rate (%)", "p-value"]:
    assert np.allclose(table[col].astype(float), official[col].astype(float), equal_nan=True), col
assert (table["95% CI for Δ"].fillna("") == official["95% CI for Δ"].fillna("")).all()
print("Matches causal_table.csv")

Matches causal_table.csv


## How much did the ablations change the generated A2?

The ablated features were active in most episodes, but the regenerated A2 is usually identical to the same SAE's
no-ablation A2. A single-token, single-layer edit may be too weak to move generation; this is the main thing to
review before reading the Δs as evidence that the features do not matter.

In [6]:
ref_a2 = rows[rows.condition == "no_ablation"].set_index(["experiment", "question_id"]).solver_attempt_2
ab = rows[rows.condition.isin(["targeted_ablation", "random_ablation"])].copy()
ab["a2_unchanged"] = ab.solver_attempt_2.to_numpy() == ref_a2.reindex(list(zip(ab.experiment, ab.question_id))).to_numpy()
display(ab.groupby(["experiment", "condition"]).agg(
    ablated_feature_active=("n_ablated_active", lambda s: (s > 0).mean()),
    mean_ablated_active=("n_ablated_active", "mean"),
    a2_unchanged=("a2_unchanged", "mean"),
).style.format({"ablated_feature_active": "{:.1%}", "mean_ablated_active": "{:.2f}", "a2_unchanged": "{:.1%}"}))